# Capesize 5TC — ML pipeline (G1 → G4)

Implements `docs/02_ML_Plan_Review.md`; decisions in `docs/decisions.md`. All logic lives in `src/capesize/`; this notebook
only runs it and shows results.

| Section | Gate |
|---|---|
| 1 Feature store + targets, point-in-time checks | G1 |
| 2 Baselines on the walk-forward (RW, seasonal, FFA-implied, ridge, ridge on the FFA residual) | G2 |
| 3 LightGBM: direct and **residual over FFA** (y − ŷ_FFA); quantiles + conformal 80% | G3 |
| 4 Model choice: rule fixed in advance (2019–2022 select, 2023–2026 confirm), main ML = ridge + LGBM-residual average | G3 |
| 5 Ablation (−FFA, −spot, −PortWatch, −fundamentals), sub-periods, importance | G4 prep |
| 6 Reference: same-day FFA (lag 0) — what the publication-time fix costs | — |
| 7 Export `ml_oos_predictions.parquet` for the text branch (+ trading-day calendar) | G4 |
| 8 Optional 3-class direction classifier (fall / flat / rise at ±0.5σ) | extra |
| 9 Extra test 2016–2018 (no PortWatch) | extra |
| 10 Stretch horizons h = 40, 65 | extra |
| 11 Results report → `outputs/reports/ML_results.md` (tables only, local) | — |

**Settings.** FFA inputs lagged one assessment day (`config.FFA_LAG = 1`: BFA publish ~17:30 London, after the 13:00 decision
time). Walk-forward: expanding, monthly refit, purge h + embargo h, test 2019-01 →. Inner validation: 3 time-ordered folds
of 250 rows, each after an h-row purge.

Setup: `pip install -r requirements.txt && pip install -e .`, set `CAPESIZE_DATA_DIR` in `.env`, kernel = repo `.venv`.
Outputs are stripped before commit (`nbstripout`); data and predictions stay in `data/` and `outputs/` (gitignored).

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from capesize import config as C
from capesize import evaluation as E
from capesize import io as cio
from capesize import models as ML
from capesize import validation as V
from capesize.checks import g1_checks, leak_control

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.precision", 4)

RERUN = False        # True: refit everything; False: reuse cached OOS predictions in data/processed/
LOG_RESULTS = False  # True: append this run to experiments/results.csv (do it on a committed, clean tree)
AUTHOR = "iluluz"
for d in (C.PROCESSED, C.TABLES, C.FIGURES):
    d.mkdir(parents=True, exist_ok=True)

def cached(name, fn):
    path = C.PROCESSED / f"{name}.parquet"
    if not RERUN and path.exists():
        return pd.read_parquet(path)
    t = time.time(); df = fn(); df.to_parquet(path, index=False)
    print(f"{name}: computed in {time.time() - t:.0f}s")
    return df

print("data dir:", cio.data_dir(), "| git:", E.git_commit(), "| FFA_LAG:", C.FFA_LAG)
cio.check_manifest()

## 1 · Feature store, targets and gate G1

In [ ]:
store, parts = E.build_store()
g, X, A, srcs, curve = (parts[k] for k in ("g", "X", "A", "srcs", "curve"))
cio.save_table(store, C.PROCESSED / "features"); cio.save_table(A, C.PROCESSED / "feature_availability")
cio.save_table(curve, C.PROCESSED / "ffa_curve_on_grid")
FEATS = ML.feature_columns(store.columns)
print(f"grid {g.index[0].date()} -> {g.index[-1].date()} ({len(g)} days) | {len(FEATS)} model features")

test = store.loc[C.TEST_START:, FEATS]
overview = pd.DataFrame({
    "group": [ML.group_of(c) for c in FEATS],
    "first_value": [store[c].first_valid_index().date() for c in FEATS],
    "nan_share_test": test.isna().mean().round(3).to_numpy(),
}, index=FEATS)
display(overview.groupby("group").agg(n=("first_value", "size"), first=("first_value", "min"),
                                      max_nan_share_test=("nan_share_test", "max")))

In [ ]:
Y = store[[c for c in store.columns if c.startswith(("y_h", "tdate_h"))]]
for k, v in g1_checks(g, X, A, srcs, Y, C.HORIZONS + C.STRETCH_HORIZONS).items():
    print(f"[G1] {v}  {k}")
print(f"[G1] negative control: a reference-period join leaks on {leak_control(g, srcs)} days (test bites)")

## 2 · Baselines (G2)

Theil's U = RMSE / RMSE(random walk). DM: HAC (h−1 Bartlett lags) + HLN correction, **stat > 0 ⇒ the benchmark is better**.
`ridge_resid` = ridge on the FFA residual, forecast = ŷ_FFA + r̂.

In [ ]:
P_base = cached("baseline_oos", lambda: E.run_baselines(store))
BASE = ["random_walk", "seasonal_naive", "ffa_implied", "ridge", "ridge_resid"]
T_base = E.metrics_table(P_base, store, BASE)
display(T_base)

## 3 · LightGBM (G3)

Per fold, training rows only: drop |corr| > 0.9; grid num_leaves {7, 15} × min_data_in_leaf {50, 100, 200} chosen by mean
early-stopped loss over 3 inner folds; refit on the whole window with the median best iteration. Huber delta = 1 sd of the
target. Monotone + on FFA basis (point model). Quantiles 0.1/0.5/0.9 and a CQR 80% interval calibrated on the last 12 months.

* `lgbm` — direct target y.
* `lgbm_resid` — target y − ŷ_FFA, forecast ŷ_FFA + r̂ (quantiles likewise). Early stopping shrinks r̂ toward the training-mean
  residual when the features carry no signal.

In [ ]:
P_lgbm = cached("lgbm_oos", lambda: pd.concat(
    [E.run_lgbm(store, h, name="lgbm", quantiles=(), conformal=False) for h in C.HORIZONS], ignore_index=True))
P_res = cached("lgbm_resid_oos", lambda: pd.concat(
    [E.run_lgbm(store, h, name="lgbm_resid", residual=True) for h in C.HORIZONS], ignore_index=True))
P0 = pd.concat([P_base, P_lgbm, P_res], ignore_index=True)
P_ens = E.average_oos(P0, ["ridge", "lgbm_resid"], "ens_ridge_lgbmres", interval_model="lgbm_resid")
P_comb = pd.concat([E.combine_oos(P0, store, h, b="lgbm_resid") for h in C.HORIZONS], ignore_index=True)
P = pd.concat([P0, P_ens, P_comb], ignore_index=True)

MODELS = BASE + ["lgbm", "lgbm_resid", "ffa_implied+lgbm_resid", "ens_ridge_lgbmres"]
T = E.metrics_table(P, store, MODELS)
display(T)
print("G3 check (Theil U < 1 and not significantly worse than FFA-implied at 5%):")
display(pd.concat([E.g3_check(T, m).assign(model=m) for m in ["lgbm", "lgbm_resid", "ens_ridge_lgbmres"]]))

In [ ]:
print("vs ridge (stat > 0 => ridge better):")
display(E.metrics_table(P, store, ["ridge", "lgbm", "lgbm_resid", "ens_ridge_lgbmres"], bench="ridge")[
    ["h", "model", "TheilU_vs_RW", "DM_vs_ridge_stat", "DM_vs_ridge_p"]])
print("Forecast encompassing, y = c + b1*FFA + b2*model (HAC):")
display(pd.concat([E.encompassing(P, h, b=m).assign(model=m) for m in ["lgbm", "ridge"] for h in C.HORIZONS]))
hp = P_res.drop_duplicates(["h", "train_end"])
print("lgbm_resid: hyper-parameters chosen (number of folds) and iterations:")
display(hp.groupby(["h", "num_leaves", "min_data_in_leaf"]).size().unstack(fill_value=0))
display(hp.groupby("h")[["n_iter", "n_features"]].describe().round(0))

## 4 · Model choice (rule fixed before looking at results)

Candidates were screened point-only; the rule: lowest mean Theil U over h = 5, 20 on **2019–2022**; **2023–2026** confirms.
Screened and not kept: recency weighting (half-life 3 years) — helps 2026 a little, worse in the selection period.

In [ ]:
SEL = ["ffa_implied", "ridge", "ridge_resid", "lgbm", "lgbm_resid", "ffa_implied+lgbm_resid", "ens_ridge_lgbmres"]
rows = []
for label, lo, hi in [("select 2019-22", "2019", "2022-12-31"), ("confirm 2023-26", "2023", "2026-12-31"),
                      ("2026 (Hormuz)", "2026", "2026-12-31"), ("all 2019-26", "2019", "2026-12-31")]:
    Tp = E.metrics_table(P[(P.date >= lo) & (P.date <= hi)], store, SEL)
    rows.append(Tp.pivot(index="model", columns="h", values="TheilU_vs_RW").assign(mean=lambda d: d.mean(axis=1))
                .add_prefix(f"{label} | "))
choice = pd.concat(rows, axis=1).loc[SEL]
display(choice.round(3))
TY = E.theil_by_year(P, ["ffa_implied", "ridge", "lgbm_resid", "ens_ridge_lgbmres"])
display(TY.round(3))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for ax, h in zip(axes, C.HORIZONS):
    d = P[(P.h == h) & (P.model == "ens_ridge_lgbmres")].set_index("date").loc["2023":]
    f = P[(P.h == h) & (P.model == "ffa_implied")].set_index("date")["yhat"].reindex(d.index)
    ax.fill_between(d.index, d.lo80, d.hi80, color="tab:blue", alpha=0.15, label="80% interval")
    ax.plot(d.index, d.y_true, color="black", lw=1, label="actual y")
    ax.plot(d.index, d.yhat, color="tab:blue", lw=1.2, label="ML (ridge + LGBM-residual)")
    ax.plot(f.index, f, color="tab:orange", lw=1, alpha=0.8, label="FFA-implied")
    ax.axhline(0, color="grey", lw=0.5); ax.set_title(f"h = {h}: ln(S_t+h / S_t), out of sample"); ax.legend(loc="upper left", ncol=4)
plt.tight_layout(); plt.savefig(C.FIGURES / "oos_ml_vs_ffa.png", dpi=120); plt.show()

cov = (P[P.model.isin(["lgbm_resid", "ens_ridge_lgbmres"])]
       .assign(inside=lambda d: (d.y_true >= d.lo80) & (d.y_true <= d.hi80))
       .groupby(["model", "h", pd.Grouper(key="date", freq="YE")])["inside"].mean().unstack([0, 1]))
cov.index = cov.index.year
print("80% interval coverage by year (target 0.75-0.85):"); display(cov.round(2))

## 5 · Ablation, importance

LightGBM-residual point model, same procedure. Benchmark = full `lgbm_resid`: **DM stat > 0 ⇒ the full model is better**
(the dropped group helps). In residual mode, `-ffa` drops the FFA *features*; the FFA base forecast stays.

In [ ]:
def run_ablation():
    return pd.concat([E.run_lgbm(store, h, name=f"lgbm_resid{n}", drop_groups=gr, residual=True,
                                 quantiles=(), conformal=False)
                      for n, gr in ML.ABLATIONS.items() for h in C.HORIZONS], ignore_index=True)

P_abl = cached("ablation_oos", run_ablation)
ABL = ["lgbm_resid"] + [f"lgbm_resid{n}" for n in ML.ABLATIONS]
PA = pd.concat([P_res, P_abl], ignore_index=True)
T_abl = E.metrics_table(PA, store, ABL, bench="lgbm_resid")
display(T_abl[["h", "model", "n", "RMSE", "TheilU_vs_RW", "DM_vs_lgbm_resid_stat", "DM_vs_lgbm_resid_p",
               "TheilU_calm", "TheilU_highvol"]])

In [ ]:
imps = {}
for h in C.HORIZONS:
    tr, te, _ = list(V.folds(store, h))[-1]
    b = store[f"ffa_hat_h{h}"].to_numpy()
    _, model, cols = ML.lgbm_fold(store[FEATS].iloc[tr], store[f"y_h{h}"].to_numpy()[tr], store[FEATS].iloc[te], h,
                                   base_tr=b[tr], base_te=b[te], quantiles=(), conformal=False, return_model=True)
    imps[f"h{h}"] = ML.gain_importance(model, cols)
imp = pd.DataFrame(imps).fillna(0).sort_values("h20", ascending=False)
print("lgbm_resid, last training window: gain importance")
display(imp.head(12).round(3))
display(imp.groupby(lambda c: ML.group_of(c)).sum().sort_values("h20", ascending=False).round(3))

## 6 · Reference: same-day FFA (lag 0)

Baselines only, with today's FFA close (a few hours after decision time). Shows how much of the earlier h = 5 result came from
the timing; not used for model choice.

In [ ]:
store0, parts0 = E.build_store(ffa_lag=0)
P_lag0 = cached("lag0_baselines_oos", lambda: E.run_baselines(store0).assign(model=lambda d: d.model + "_lag0"))
T_lag0 = E.metrics_table(P_lag0, store0, ["ffa_implied_lag0", "ridge_lag0", "ridge_resid_lag0"], bench="ffa_implied_lag0")
cmp = (T[T.model.isin(["ffa_implied", "ridge", "ridge_resid"])][["h", "model", "TheilU_vs_RW"]]
       .merge(T_lag0.assign(model=T_lag0.model.str.replace("_lag0", ""))[["h", "model", "TheilU_vs_RW"]],
              on=["h", "model"], suffixes=("_lag1", "_lag0")))
display(cmp)

## 7 · Export (G4 interface) and logging

**Frozen as `ml-v1.0` (2026-11-08):** main ML = `ens_ridge_lgbmres`. Ridge had the lowest 2019–2022 mean U (0.911 vs 0.923) but
gives no quantiles, which the hand-over (C7) needs; ridge is reported alongside as an equally strong linear benchmark.

In [ ]:
MAIN = "ens_ridge_lgbmres"
version = "ml-v1.0"   # G4 freeze 2026-11-08 (docs/decisions.md), git tag ml-v1.0
oos = E.oos_export(P, MAIN, version)
oos.to_parquet(C.PROCESSED / "ml_oos_predictions.parquet", index=False)
P.to_parquet(C.PROCESSED / "all_oos_predictions.parquet", index=False)
T.to_csv(C.TABLES / "model_metrics.csv", index=False)
choice.to_csv(C.TABLES / "model_choice.csv")
T_abl.to_csv(C.TABLES / "ablation_metrics.csv", index=False)
TY.to_csv(C.TABLES / "theil_by_year.csv", index=False)
cmp.to_csv(C.TABLES / "ffa_lag_reference.csv", index=False)
print(oos.shape, oos.columns.tolist()); display(oos.tail(3))

# Baltic 5TC publication dates (dates only) for the text branch's news-window merge (--trading-days)
from capesize import data as D
td = D.load_spot().dropna(subset=["S"]).index
pd.DataFrame({"date": td.strftime("%Y-%m-%d")}).to_csv(C.PROCESSED / "baltic_5tc_trading_days.csv", index=False)
print(f"trading days: {len(td)} ({td[0].date()} -> {td[-1].date()}) -> data/processed/baltic_5tc_trading_days.csv")

if LOG_RESULTS:
    E.append_results(T[T.model.isin(["ridge", "ridge_resid", "lgbm", "lgbm_resid", MAIN])], P, AUTHOR, notes="ffa_lag=1")
    E.append_results(T_abl[T_abl.model != "lgbm_resid"], PA, AUTHOR, notes="ablation, lgbm_resid point only")

## 8 · Optional 3-class direction classifier (docs/02 3.4)

LightGBM multiclass on `cls_h` (fall / flat / rise at ±0.5σ_h, σ from past targets only), same walk-forward, correlation
filter, grid and inner folds as section 3. Compared with the climatology (training class frequencies / majority class) and
with regression forecasts mapped to classes at the same thresholds. DM on per-row log loss: **stat > 0 ⇒ climatology better**.

In [ ]:
P_cls = cached("cls_oos", lambda: pd.concat([E.run_lgbm_cls(store, h) for h in C.HORIZONS], ignore_index=True))
T_cls = E.class_table(P_cls, P, store, ["ffa_implied", "ridge", "ens_ridge_lgbmres"])
display(T_cls.round(3))
T_cls.to_csv(C.TABLES / "direction_classifier.csv", index=False)

## 9 · Extra test 2016–2018 (docs/02 3.6, "if time allows")

Same procedure, test blocks 2016-01 → 2018-12, PortWatch group dropped (it starts 2019). Early folds have only ~1.5–3 years of
training data, so the inner validation uses the small-sample fallback (`models.inner_splits`: 3 blocks of (n − 150 − h)/3 rows).
Not used for model choice.

In [ ]:
XW = dict(test_start="2016-01-01", test_end=C.TEST_START)
P_x = cached("extra2016_oos", lambda: pd.concat(
    [E.run_baselines(store, **XW)]
    + [E.run_lgbm(store, h, name="lgbm_resid", residual=True, drop_groups=("portwatch",), quantiles=(),
                  conformal=False, **XW) for h in C.HORIZONS], ignore_index=True))
P_x = pd.concat([P_x, E.average_oos(P_x, ["ridge", "lgbm_resid"], "ens_ridge_lgbmres")], ignore_index=True)
XM = ["random_walk", "seasonal_naive", "ffa_implied", "ridge", "ridge_resid", "lgbm_resid", "ens_ridge_lgbmres"]
COLS = ["h", "model", "n", "MAE", "TheilU_vs_RW", "hit_rate", "F1_3class", "DM_vs_ffa_implied_stat", "DM_vs_ffa_implied_p",
        "TheilU_calm", "TheilU_highvol"]
T_x = E.metrics_table(P_x, store, XM)
display(T_x[COLS])
display(pd.concat([E.g3_check(T_x, m).assign(model=m) for m in ["lgbm_resid", "ens_ridge_lgbmres"]]))
TY_x = E.theil_by_year(P_x, ["ffa_implied", "ridge", "lgbm_resid", "ens_ridge_lgbmres"])
display(TY_x.round(3))
T_x.to_csv(C.TABLES / "extra_test_2016_2018.csv", index=False)

## 10 · Stretch horizons h = 40, 65 (docs/02 3.2)

Same walk-forward (purge h + embargo h, so training stops 80 / 130 rows before each test month), test 2019 →. FFA-implied adds
the M+3 node (h = 65 is ~3 months out). Caution: ~12 and ~4 independent targets a year, so DM tests (HAC, h − 1 lags) have
little power and are approximate.

In [ ]:
st = E.with_stretch_ffa(store, curve)
SH = C.STRETCH_HORIZONS
P_s = cached("stretch_oos", lambda: pd.concat(
    [E.run_baselines(st, horizons=SH)]
    + [E.run_lgbm(st, h, name="lgbm_resid", residual=True, quantiles=(), conformal=False) for h in SH],
    ignore_index=True))
P_s = pd.concat([P_s, E.average_oos(P_s, ["ridge", "lgbm_resid"], "ens_ridge_lgbmres")], ignore_index=True)
T_s = E.metrics_table(P_s, st, XM, horizons=SH)
display(T_s[COLS])
TY_s = E.theil_by_year(P_s, ["ffa_implied", "ridge", "lgbm_resid", "ens_ridge_lgbmres"], horizons=SH)
display(TY_s.round(3))
T_s.to_csv(C.TABLES / "stretch_h40_h65.csv", index=False)

## 11 · Results report

Writes `outputs/reports/ML_results.md` (metrics tables only, no prices; `outputs/` is gitignored). Open it in VS Code with
Markdown preview (Ctrl+Shift+V).

In [ ]:
def mdt(df, digits=3):
    return df.round(digits).to_markdown(index=False)

main_cols = ["h", "model", "n", "TheilU_vs_RW", "hit_rate", "F1_3class", "DM_vs_ffa_implied_stat", "DM_vs_ffa_implied_p",
             "TheilU_calm", "TheilU_highvol", "pinball", "coverage80", "width80"]
g3 = pd.concat([E.g3_check(T, m).assign(model=m) for m in ["lgbm", "lgbm_resid", "ens_ridge_lgbmres"]])
vs_ridge = E.metrics_table(P, store, ["ridge", "lgbm", "lgbm_resid", "ens_ridge_lgbmres"], bench="ridge")
sections = [
    ("Settings", f"git `{E.git_commit()}` · FFA_LAG = {C.FFA_LAG} · test {C.TEST_START} → {P.date.max().date()} · "
                 "expanding walk-forward, monthly refit, purge h + embargo h · main ML = `ens_ridge_lgbmres` "
                 "(equal-weight ridge + LightGBM on the FFA residual; interval from `lgbm_resid` shifted). "
                 "Theil U < 1 beats the random walk; DM stat > 0 ⇒ the benchmark is better."),
    ("1. Main table (2019 →, benchmark FFA-implied)", mdt(T[main_cols])),
    ("2. Gate G3", mdt(g3)),
    ("3. Against ridge (DM stat > 0 ⇒ ridge better)", mdt(vs_ridge[["h", "model", "TheilU_vs_RW", "DM_vs_ridge_stat",
                                                                     "DM_vs_ridge_p"]])),
    ("4. Model choice (rule: mean U over h on 2019–2022; 2023–2026 confirms)",
     choice.round(3).reset_index().to_markdown(index=False)),
    ("5. Theil U by year", mdt(TY)),
    ("6. 80% interval coverage by year (target 0.75–0.85)",
     cov.round(2).set_axis([f"{m} h{h}" for m, h in cov.columns], axis=1).reset_index().to_markdown(index=False)),
    ("7. Ablation (lgbm_resid; DM stat > 0 ⇒ full model better)",
     mdt(T_abl[["h", "model", "TheilU_vs_RW", "DM_vs_lgbm_resid_stat", "DM_vs_lgbm_resid_p"]])),
    ("8. Gain importance by group (lgbm_resid, last window)", imp.groupby(lambda c: ML.group_of(c)).sum()
     .sort_values("h20", ascending=False).round(3).reset_index().to_markdown(index=False)),
    ("9. FFA timing reference (lag 1 used vs same-day lag 0)", mdt(cmp)),
    ("10. 3-class direction classifier", mdt(T_cls)),
    ("11. Extra test 2016–2018 (no PortWatch)", mdt(T_x[COLS])),
    ("12. Stretch horizons h = 40, 65", mdt(T_s[COLS])),
]
(C.TABLES.parent / "reports").mkdir(exist_ok=True)
rep = C.TABLES.parent / "reports" / "ML_results.md"
nl = chr(10)
rep.write_text(f"# Capesize 5TC — ML results{nl}{nl}" + (nl * 2).join(f"## {t}{nl}{nl}{b}" for t, b in sections) + nl,
               encoding="utf-8")
print("wrote", rep)